# SEC MAIS — Teste e Inspeção Bronze: Mais Estudos

Este notebook inspeciona os dados brutos extraídos do programa **Mais Estudos** armazenados em `armazenamento/bronze/mais_estudos/`.

In [ ]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Localiza o diretório do Mais Estudos na camada bronze
def obter_caminho_mais_estudos() -> Path:
    candidatos = [
        Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "mais_estudos",
        Path.cwd().parent / "sec-mais" / "armazenamento" / "bronze" / "mais_estudos",
        Path.cwd() / "armazenamento" / "bronze" / "mais_estudos",
        Path.cwd().parent / "armazenamento" / "bronze" / "mais_estudos",
    ]
    for c in candidatos:
        if c.exists():
            return c.resolve()
    return (Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "mais_estudos").resolve()

caminho_me = obter_caminho_mais_estudos()
print(f"📁 Caminho Mais Estudos: {caminho_me}")
arquivos = list(caminho_me.glob("**/*.parquet")) if caminho_me.exists() else []
print(f"📄 Arquivos Parquet encontrados: {len(arquivos)}")
for arq in arquivos:
    print(f"   - {arq.name} ({round(arq.stat().st_size / 1024, 2)} KB)")

## 1. Carregamento dos Dados via Pandas

In [ ]:
if arquivos:
    arquivo_mais_recente = sorted(arquivos, key=lambda f: f.stat().st_mtime, reverse=True)[0]
    print(f"Lendo arquivo: {arquivo_mais_recente.name}")
    
    df = pd.read_parquet(arquivo_mais_recente)
    display(df.head(10))
else:
    print("⚠️ Nenhum arquivo parquet encontrado para o Mais Estudos.")
    print("Para extrair os dados via VPN Oracle, execute no terminal:")
    print("  python -m extracao.programas.mais_estudos")
    df = pd.DataFrame()

## 2. Estrutura, Tipagem e Verificação de Nulos

In [ ]:
if not df.empty:
    print(f"Total de Linhas: {len(df)}")
    print(f"Total de Colunas: {len(df.columns)}\n")
    
    info_colunas = pd.DataFrame({
        "Tipo": df.dtypes,
        "Valores Não-Nulos": df.notnull().sum(),
        "Nulos": df.isnull().sum(),
        "% Nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "Valores Únicos": df.nunique()
    })
    display(info_colunas)
else:
    print("Sem dados para analisar.")

## 3. Validação de Metadados Técnicos (`_data_extracao`, `_programa`)

In [ ]:
if not df.empty:
    cols_tecnicas = [c for c in df.columns if c.startswith("_")]
    print(f"Colunas técnicas de controle: {cols_tecnicas}")
    if cols_tecnicas:
        display(df[cols_tecnicas].drop_duplicates())
else:
    print("Sem dados para analisar.")